In [ ]:
# !pip install tensorflow
# !pip install pandas
# !pip install numpy
# !pip install matplotlib
# !pip install sklearn

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib import pyplot
from tensorflow.keras import Model
from tensorflow.keras import Sequential
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dropout, Activation, Dense
from tensorflow.keras.optimizers import Adadelta
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import Dense
# from tensorflow.keras.layers import LSTM
import sklearn
from sklearn.preprocessing import MinMaxScaler

In [ ]:
filepath = "datasets/iot23-1.csv"
df = pd.read_csv(filepath)

In [ ]:
df.head(10)

In [ ]:
# Count the unique values in the "ColumnName" column
vc = df["label"].value_counts()

# Display the counts
print(vc)

In [ ]:
print(df.columns)

In [ ]:
# Sort the DataFrame by timestamp (paper Bram Van Dartel)
df.sort_values(by='ts', inplace=True)

# Calculate inter-arrival time and store in a new column
df['time_diff'] = df['ts'].diff()

In [ ]:
# Drop columns with weak correlation
columns_to_drop = ['Unnamed: 0','ts', 'uid', 'id.orig_h', 'local_orig', 'local_resp', 'missed_bytes','history','id.resp_h','service','id.orig_p','id.resp_p']
df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
print(df.columns)

In [ ]:
# manually encode label values to 0 or 1
from sklearn.preprocessing import LabelEncoder

# Sample DataFrame with 'label' column
data = {'label': ['PartOfAHorizontalPortScan', 'Okiru', 'Benign', 'DDoS', 'C&C', 'C&C-HeartBeat']}
labeldata = pd.DataFrame(data)

# Create a LabelEncoder instance
label_encoder = LabelEncoder()

# Fit the LabelEncoder and transform the 'label' column
df['label'] = label_encoder.fit_transform(df['label'])

# Convert 'Benign' label to 0 and others to 1
df['label'] = df['label'].apply(lambda x: 0 if x == label_encoder.transform(['Benign'])[0] else 1)


In [ ]:
df.head(100)

In [ ]:
#mencari data kosong
df[df.isnull().any(axis=1)]

In [ ]:
#mengisi data kosong dengan angka 0
df=df.fillna({'time_diff':'0'})

In [ ]:
#find and replace '-' with 0
df['duration'] = df['duration'].replace('-',0)
df['orig_bytes'] = df['orig_bytes'].replace('-',0)
df['resp_bytes'] = df['resp_bytes'].replace('-',0)

In [ ]:
df.head()

In [ ]:
#OneHotEncoding terhadap kolom proto dan conn_state
df = pd.get_dummies(df, columns=["proto","conn_state"])

In [ ]:
df.head()

In [ ]:
# Filter columns with boolean values
boolean_columns = df.select_dtypes(include=['bool']).columns

# Print the columns with boolean values
print(boolean_columns)

In [ ]:
#convert boolean value to int
columns_to_convert = ['proto_icmp', 'proto_tcp', 'proto_udp', 'conn_state_OTH',
       'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0',
       'conn_state_RSTR', 'conn_state_S0', 'conn_state_S1',
       'conn_state_S2', 'conn_state_SF']
df[columns_to_convert] = df[columns_to_convert].astype(int)

In [ ]:
df.head()

In [ ]:
print(df.columns)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Create a MinMaxScaler instance
scaler = MinMaxScaler()

# Fit and transform the selected columns
columns_to_normalize = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'orig_ip_bytes', 
                        'resp_pkts', 'resp_ip_bytes', 'time_diff', 'proto_icmp','proto_tcp', 'proto_udp', 
                        'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0', 
                        'conn_state_RSTR', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2', 'conn_state_SF']

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

# Print the normalized DataFrame
# print(df)

In [ ]:
df.head(10)

In [ ]:
print(df.columns)

In [ ]:
X = df[['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'orig_ip_bytes',
       'resp_pkts', 'resp_ip_bytes', 'time_diff', 'proto_icmp',
       'proto_tcp', 'proto_udp', 'conn_state_OTH', 'conn_state_REJ',
       'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2', 'conn_state_SF',]].values

In [ ]:
X.shape

In [ ]:
Y = df[['label']].values

In [ ]:
Y.shape

In [ ]:
X

In [ ]:
Y

In [ ]:
df['label'].value_counts()

In [ ]:
#split data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=10, test_size=0.2)

In [ ]:
X_train.shape

In [ ]:
#model layer
model = Sequential()
model.add(Dense(500, activation='relu',input_dim=20))
model.add(Dense(200, activation='relu'))
model.add(Dropout(0.2))
# model.add(Dense(400,activation='relu'))
# model.add(Dropout(0.2))
# model.add(Dense(200,activation='relu'))
# model.add(Dropout(0.2))
# model.add(Dense(150,activation='relu'))
# model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
#model compile
from keras.optimizers import Adam

learning_rate = 0.001
optimizer = Adam(learning_rate=learning_rate)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
#early stopping callbacks
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from keras.models import load_model

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
mc = ModelCheckpoint('best_model1.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)

In [ ]:
model.summary()

In [ ]:
import time
start = time.time()
print('program start...')
print()

history = model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs = 20, batch_size=512,  verbose=1, callbacks=[es, mc])

print()
end = time.time()
print('program end...')
print()
print('time cost: ')
print(end - start, 'seconds')

In [ ]:
# load the saved model
saved_model = load_model('best_model1.h5')
# evaluate the model
_, train_acc = model.evaluate(X_train, Y_train, verbose=0)
_, test_acc = model.evaluate(X_test, Y_test, verbose=0)
print('Train: %.3f, Test: %.3f' % (train_acc, test_acc))
# plot training history
pyplot.plot(history.history['accuracy'], label='train')
pyplot.plot(history.history['val_accuracy'], label='test')
pyplot.legend()
pyplot.show()

In [ ]:
#report imbalanced
Y_pred1 = model.predict(X_test)
Y_pred_classes1 = (Y_pred > 0.5).astype(int)

from sklearn.metrics import classification_report

report = classification_report(Y_test, Y_pred_classes1)
print(report)

In [ ]:
#metrics imbalanced
from sklearn.metrics import accuracy_score, recall_score, precision_score

accuracy = accuracy_score(Y_test, Y_pred_classes1)
recall = recall_score(Y_test, Y_pred_classes1)
precision = precision_score(Y_test, Y_pred_classes1)

print(f"accuracy= {accuracy}")
print(f"recall= {recall}")
print(f"precision= {precision}")

In [ ]:
#SMOTE
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=12, sampling_strategy = 1.0)
x_train_res, y_train_res = sm.fit_resample(X_train, Y_train)
x_test_res, y_test_res = sm.fit_resample(X_test, Y_test)

from collections import Counter
print(Counter(y_train_res))

In [ ]:
#tunig learning_rate
learning_rate = 0.00001
optimizer = Adam(learning_rate=learning_rate)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

#callbacks
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
mc = ModelCheckpoint('best_model1.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)

In [ ]:
import time
start = time.time()
print('program start...')
print()

history2 = model.fit(x_train_res, y_train_res, validation_data=(x_test_res, y_test_res), epochs = 20, batch_size=512,  verbose=1, callbacks=[es, mc])

print()
end = time.time()s
print('program end...')
print()
print('time cost: ')
print(end - start, 'seconds')

In [ ]:
#report SMOTE
Y_pred2 = model.predict(x_test_res)
Y_pred_classes2 = (Y_pred2 > 0.5).astype(int)

from sklearn.metrics import classification_report

report2 = classification_report(y_test_res, Y_pred_classes2)
print(report)

In [ ]:
#metric
from sklearn.metrics import accuracy_score

accuracy2 = accuracy_score(y_test_res, Y_pred_classes2)
recall2 = recall_score(y_test_res, Y_pred_classes2)
precision2 = precision_score(y_test_res, Y_pred_classes2)

print(f"accuracy= {accuracy2}")
print(f"recall= {recall2}")
print(f"precision= {precision2}")

In [ ]:
# load the saved model
saved_model = load_model('best_model1.h5')
# evaluate the model
_, train_acc = model.evaluate(x_train_res, y_train_res, verbose=0)
_, test_acc = model.evaluate(x_test_res, y_test_res, verbose=0)
print('Train: %.3f, Test: %.3f' % (train_acc, test_acc))
# plot training history
pyplot.plot(history2.history['accuracy'], label='train')
pyplot.plot(history2.history['val_accuracy'], label='test')
pyplot.legend()
pyplot.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define the performance metrics for both datasets
labels = ['Balanced', 'Imbalanced']
accuracy_balanced = accuracy2
recall_balanced = recall2
precision_balanced = precision2

accuracy_imbalanced = accuracy
recall_imbalanced = recall
precision_imbalanced = precision

# Set the width of the bars
bar_width = 0.2

# Set the x-axis positions for bars
index = np.arange(len(labels))

# Create the figure and axes for subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# First subplot (Accuracy and Recall)
# Plot accuracy
ax1.bar(index - bar_width, [accuracy_balanced, accuracy_imbalanced], bar_width, label='Accuracy')
# Plot recall
ax1.bar(index, [recall_balanced, recall_imbalanced], bar_width, label='Recall')

# Set the x-axis labels for the first subplot
ax1.set_xticks(index)
ax1.set_xticklabels(labels)

# Set the y-axis label for the first subplot
ax1.set_ylabel('Metrics')

# Set the title for the first subplot
ax1.set_title('Accuracy and Recall Comparison')

# Add a legend for the first subplot
ax1.legend()

# Second subplot (Precision)
# Set the x-axis positions and labels
labels_precision = ['Class 0', 'Class 1']
x_precision = range(len(labels_precision))

# Plot precision for both datasets
bar_width_precision = 0.35
ax2.bar(x_precision, precision_balanced, width=bar_width_precision, label='Balanced Dataset')
ax2.bar([i + bar_width_precision for i in x_precision], precision_imbalanced, width=bar_width_precision, label='Imbalanced Dataset')

# Set the x-axis labels for the second subplot
ax2.set_xticks([i + bar_width_precision / 2 for i in x_precision])
ax2.set_xticklabels(labels_precision)

# Add a legend for the second subplot
ax2.legend()

# Set the title and labels for the second subplot
ax2.set_title('Precision Comparison')
ax2.set_xlabel('Class')
ax2.set_ylabel('Precision')

# Show or save the plot
plt.tight_layout()
plt.show()
